In [ ]:
import time
import dash
from dash import dcc, html, Input, Output, State

app = dash.Dash(__name__)

app.layout = html.Div([
    html.Button("Start", id="btn", n_clicks=0),
    html.Div(id="timer-display", style={"fontFamily": "monospace", "marginTop": "8px"}),
    dcc.Interval(id="interval", interval=200, disabled=True),
    dcc.Store(id="t-start", data=None),
    dcc.Store(id="t-end",   data=None),
])

# 1. Record start time client-side the instant the button is clicked
app.clientside_callback(
    "function(n) { return Date.now() / 1000.0; }",
    Output("t-start", "data"),
    Input("btn", "n_clicks"),
    prevent_initial_call=True,
)

# 2. Clear end time on click so stale value doesn't flash
app.clientside_callback(
    "function(n) { return null; }",
    Output("t-end", "data", allow_duplicate=True),
    Input("btn", "n_clicks"),
    prevent_initial_call=True,
)

# 3. Enable interval on click, disable it when t-end is written
app.clientside_callback(
    """
    function(n, t_end) {
        const ctx = dash_clientside.callback_context;
        if (!ctx.triggered.length) return window.dash_clientside.no_update;
        if (ctx.triggered[0].prop_id === 't-end.data') return true;
        return false;
    }
    """,
    Output("interval", "disabled"),
    [Input("btn", "n_clicks"), Input("t-end", "data")],
    prevent_initial_call=True,
)

# 4. Update display on every tick and when t-end arrives
app.clientside_callback(
    """
    function(n, t_end, t_start) {
        if (!t_start) return '';
        if (t_end !== null && t_end !== undefined && t_end >= t_start)
            return '\u2713 ' + (t_end - t_start).toFixed(1) + 's';
        return '\u23f1 ' + (Date.now() / 1000.0 - t_start).toFixed(1) + 's';
    }
    """,
    Output("timer-display", "children"),
    [Input("interval", "n_intervals"), Input("t-end", "data")],
    State("t-start", "data"),
)

# 5. Your slow work goes here — return time.time() as the last output
@app.callback(
    [Output("timer-display", "children", allow_duplicate=True),
     Output("t-end", "data")],
    Input("btn", "n_clicks"),
    prevent_initial_call=True,
)
def do_work(_n):
    time.sleep(3)           # replace with your actual work
    elapsed = "done"
    return elapsed, time.time()


if __name__ == "__main__":
    app.run(debug=True, port=8051)


OSError: Address 'http://127.0.0.1:8050' already in use.
    Try passing a different port to run.